## **Welcome to the Lostma Workshop!**

This Notebook aims to explore the data collected as part of the project.

# **I) Notebook setup**

A first release of the data is already available: [https://zenodo.org/records/18793812](https://zenodo.org/records/18793812)

You can find all the data on [Heurist](https://heurist.huma-num.fr/heurist/?db=jbcamps_gestes) and the documentation on our [website](https://lostma-erc.github.io/corpus/documentation)

![data-pilepilne](./images/data_pipeline.png)

To use this notebook, you first need to download the library created in order to facilitate exploration of our Heurist Database. This tool is based on the [Heurist-API](https://pypi.org/project/heurist-api) developped by Kelly Christensen. Thanks to this software, we can extract, transform and load the content of an Heurist database into a DuckDB database making it easier to use queries with SQL. We then created a specific LostMa Python object to help us to explore the content of the data.

So please, start with installation:

In [ ]:
!pip install "git+https://github.com/LostMa-ERC/Heurist-analyser.git"

You can then use your Heurist login details to download the Lostma database:

In [ ]:
# create the client object
from lostma_db import LostmaDB

login = "your login"
pwd = "your password" 
db = LostmaDB(login, pwd)

In [ ]:
# download the db and its schema
db.sync()

# **II) Scope of the 1st data release**

Here are the languages listed in the db and the current count of entities for each:

| Language | dum (Middle Dutch) | enm (Middle English) | non_WEST (West Old Norse) | non_EAST (East Old Norse) | frm (Middle French) | fro (Old French) | fro_PRO (Franco-Occitan) | pro (Occitan) | frp (Franco-Provençal) | fro_ENG (Anglo-Norman) | fro_ITA (Franco-Italian) | ita (Italian) | lat (Latin) | gmh (Middle High German) | gml (Middle Low German) | cat (Catalan) | glg (Galician) | glg_POR (Galician-Portugese) | por (Portugese) | spa (Spanish) | ghg (Early Modern Irish) | mga (Middle Irish) | oco (Old Cornish) | wlm (Middle Welsh) |
|:--------|--------:|--------:|--------:|--------:|--------:|--------:|--------:|--------:|--------:|--------:|--------:|--------:|--------:|--------:|--------:|--------:|--------:|--------:|--------:|--------:|--------:|--------:|--------:|--------:|
|N. texts | 78   | 69   | 79    | 0     | 90   | 280    | 2     | 17   |   2  |  25  |20     | 1 | 17   | 6    | 0     | 0   | 0    | 0     | 0   | 0    |0    | 0   | 0    |0  |
|N. wits | 170     | 172   | 58    | 0     | 240   | 1807    | 6     | 16   |   2  |  63  | 31    | 0   | 0    | 0     | 0   | 0    | 0     | 0   | 0    |0    | 0   | 0    |0    | 0 |

For this release, we decided to focus on 6 languages that already contained widely completed data:

In [ ]:
available_languages = ["dum (Middle Dutch)", "enm (Middle English)", "fro (Old French)", "frm (Middle French)", "fro_ENG (Anglo-Norman)", "fro_ITA (Franco-Italian)"]

With this scope, you can use the following scripts to generate the 3 tables that bring together all the data needed to study the tradition of texts:

In [ ]:
# data from the Witness, Text, Story and Genre tables
witnesses = db.witnesses(available_languages)
witnesses

As you can see, a filter is removing fields that are filled in less than 5% of the rows in order to avoid over-empty columns. But some of them are still usefull.

In [ ]:
# fields we want to keep for the release
always_keep_these_columns = ["TextTable_is_adapted_by H-ID", "TextTable_is_adapted_by Name",
                             "TextTable_place_of_creation H-ID", "TextTable_place_of_creation Name"]

In [ ]:
witnesses = db.witnesses(available_languages, always_keep_these_columns)
witnesses
# NB: it is also possible to add another threshold as the third argument of the function.

In [ ]:
# to download the result as a CSV file
witnesses.to_csv("witnesses.tsv", sep="\t", index=False)

In [ ]:
# data from the Part, Document, Repository and Digitization tables
parts = db.parts(available_languages)
parts

In [ ]:
# specific relational data from the Story and Storyverse tables
stories = db.stories(available_languages)
stories

In [ ]:
# you can then repeat the steps, adding all the languages and fields you want to include in the result tables
available_languages += ["non_WEST (West Old Norse)"]

# **III) Data completeness**

The following scripts had been created to help make choices defining the scope of the release

In [ ]:
# load an overview of the db
overview = db.overview()

In [ ]:
# style the output with a color code based on the data completeness
import pandas as pd

raw = overview.copy()
counts = raw.loc["nbr_witnesses"]
data = raw.drop(index=["nbr_witnesses"])
if data.max().max() <= 1.0:
    data = data * 100  # en %

data_full = pd.concat([counts.to_frame().T, data])

def color_by_threshold(v):
    if pd.isna(v):
        return ""
    v = float(v)
    v = max(0.0, min(100.0, v))

    if v < 30:
        color = "#FF0000"  # rouge
    elif v < 70:
        color = "#ff8000"  # orange clair
    elif v < 95:
        color = "#FFFF00" # jaune clair
    else:
        color = "#90EE90"  # vert

    return f"background: linear-gradient(90deg, {color} {v}%, white {v}%); border: 1px solid #eee;"

styled = (
    data_full.style
      .format("{:.0f}%", subset=pd.IndexSlice[data_full.index != "nbr_witnesses", :])
      .format("{:.0f}", subset=pd.IndexSlice[data_full.index == "nbr_witnesses", :])
      .map(color_by_threshold, subset=pd.IndexSlice[data_full.index != "nbr_witnesses", :])
      .set_table_styles([
            {"selector": "th.row_heading", "props": [("text-align", "right")]}
        ])
)

In [ ]:
# fancy dataviz
from lostma_db import dataviz_overview
always_visible = {"nbr_witnesses"}

sections = [
    {"id": "summary", "label": "Résumé", "match": lambda idx: idx in always_visible},
    {"id": "witness", "label": "Witness", "match": lambda idx: str(idx).startswith("Witness_")},
    {"id": "text", "label": "Text", "match": lambda idx: str(idx).startswith("TextTable_")},
    {"id": "genre", "label": "Genre", "match": lambda idx: str(idx).startswith("Genre_")},
    {"id": "story", "label": "Story", "match": lambda idx: str(idx).startswith("Story_")},
    {"id": "part", "label": "Part", "match": lambda idx: str(idx).startswith("Part_")},
    {"id": "document", "label": "Document", "match": lambda idx: str(idx).startswith("DocumentTable_")},
    {"id": "repository", "label": "Repository", "match": lambda idx: str(idx).startswith("Repository_")},
    {"id": "digitization", "label": "Digitization", "match": lambda idx: str(idx).startswith("Digitization_")}
]

dataviz_overview(
    styler=styled,
    title="Overview of data collected by the LostMa team",
    sections=sections,
    default_open={"witness"}
)

Nota Bene : For this release, we decided to eliminate a certain amount of data by default, including data from the PhysDesc table. Moreover, the witnesses and parts tables exclude isolated data that is not linked to either of these two tables.

In [ ]:
# load a specific overview for a given specific table and language
analyse = db.analyse("witness", "dum (Middle Dutch)")

In [ ]:
# number of entries and data entries with action required
analyse["summary"]

In [ ]:
# completeness table for each attribute
analyse["data"]

A cartography of the LostMa’s Heurist database content is available here : https://docs.google.com/document/d/1pXC03PCSUtxR0RqdYXRp69rdkAK-HIZaSGyxlWWZMZQ/edit?tab=t.0

# Penser à dire que tout ça se place du point de vue de la table d'origine (witness) et que le résultat n'est donc pas complet !

In [ ]:
db.sql("SELECT * FROM witness")

NB: If you know [SQL](https://www.w3schools.com/sql/), you can use the [data model](https://github.com/LostMa-ERC/DataArchitect2025/blob/main/heurist.jpg) to extract all the data you want to see as a dataframe

To help you, we created some pre-defined queries to explore the most usefull data (we recommend limiting the scope to ready-to-use language corpora)

In [ ]:
available_languages = ["dum (Middle Dutch)", "enm (Middle English)", "non_WEST (West Old Norse)", "fro (Old French)", "frm (Middle French)", "fro_ENG (Anglo-Norman)", "fro_ITA (Franco-Italian)"]

In [ ]:
db.texts(available_languages)

In [ ]:
witnesses = db.witnesses(available_languages)
witnesses

NB : The witnesses output contains also the data from the text, part and document tables

All the results of these functions can be donwloaded as a csv file for closer examination:

In [ ]:
witnesses.to_csv("witnesses.tsv", sep="\t", index=False)

In [ ]:
parts = db.parts(available_languages)
parts

In [ ]:
stories = db.stories()
stories

As the date of texts and witnesses is not always known precisely, the temporal data is entered in the form of a dictionary with several entries.
If you want to apply a time interval to a temporal attribute, you can use this specific function on any output from the previous functions :

In [ ]:
filter_by_interval(witnesses, "date_of_creation", 1225, 1265)

If you want to know the boundary of a temporal attribute:

In [ ]:
temporal_extent(witnesses, "date_of_creation")

If you want to study the tradition of texts (e. g. with [siMAtree](https://github.com/LostMa-ERC/simMAtree)), you can use this function to have an abundance distribution data output:

In [ ]:
db.tradition(available_languages).to_csv("tradition.csv", index=False)

![data-model](./images/data_model.png)

# III) **Data visualisation**

Once you get the data you need, you can explore them with dedicated software

In [ ]:
import pandas as pd
import plotly.express as px
import numpy as np

In [ ]:
df_texts = db.texts(available_languages)
vc_texts = df_texts["language_COLUMN"].value_counts()
counts_texts = pd.DataFrame({
    "language": vc_texts.index,
    "count": vc_texts.values,
    "source": "texts",
})
df_witnesses = db.witnesses(available_languages)
vc_witnesses = df_witnesses["language_COLUMN"].value_counts()
counts_witnesses = pd.DataFrame({
    "language": vc_witnesses.index,
    "count": vc_witnesses.values,
    "source": "witnesses",
})

In [ ]:
counts_all = pd.concat([counts_texts, counts_witnesses], ignore_index=True)
fig = px.bar(
    counts_all,
    x="language",
    y="count",
    color="source",
    barmode="group",
)
fig.show()

If you want to know the data available for a corpus for a specific table, we created a function to see:

- the "completeness table" of each field
- the number of "total records" for this scope
- how many records have the "action required" field open

In [ ]:
db.analyse("document", "frm (Middle French)")['completeness table']

You can then see how the data is distributed within a single field:

In [ ]:
attr_col = "status_witness"

df_witnesses = db.witnesses(available_languages)
s = df_witnesses[attr_col]
if s.apply(lambda x: isinstance(x, np.ndarray)).any():
    s = s.explode()
vc_attr = s.value_counts()
counts_attr = pd.DataFrame({
    "value": vc_attr.index,
    "count": vc_attr.values,
})

In [ ]:
fig = px.bar(
    counts_attr,
    x="value",
    y="count",
)
fig.update_layout(
    xaxis_title=attr_col,
    yaxis_title="Number of records",
    xaxis_tickangle=-45,
)
fig.show()

Or how this field is distributed according to language:

In [ ]:
lang_col = "language_COLUMN"

grouped = (
    df_witnesses
    .groupby([lang_col, attr_col])
    .size()
    .reset_index(name="count")
)

fig = px.bar(
    grouped,
    x=lang_col,
    y="count",
    color=attr_col,
    barmode="group"
)

fig.update_layout(
    title=f"Répartition des valeurs de {attr_col} par langue",
    xaxis_title="Langue",
    yaxis_title="Nombre de textes",
    xaxis_tickangle=-45,
)
fig.show()

# IV) **Beyond Heurist**

We are currently working on a way to combine the metadata about texts with their transcription and data available on [OpenStemmata](https://openstemmata.github.io/).  The idea is to make all the data accessible through a general RESTful API using DTS standards. For now, the metadata on the texts are already accessible in XML-TEI format in a [Github repository](https://github.com/LostMa-ERC/tei-depot) sorted by language.

If you are looking for a specific group of texts using their ID, please use this function to download the files :

In [ ]:
from lostma_db import download_text_in_tei
download_text_in_tei(50224)